# HBOT Treatment Target Knowledge Assembly on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kgrid-objects/FAIR-DO-Workshop/blob/main/HBOT3-KA/auxiliary/aux-notebook/hbot_treatment_target_ka_colab.ipynb)

This notebook clones the repository and runs the real, governed KA implementation from `src/orchestrator.js`, which orchestrates the four constituent Knowledge Objects (Wagner, HBOT Decision, Burden, Margolis) exactly as implemented in this package — no reimplemented logic.

Because Colab cells cannot answer live interactive prompts, the Wagner and Burden questionnaires (normally collected through their own interactive Questionnaire Logic capability) are answered here with scripted `wagnerAskYesNo` / `burdenAskQuestion` callbacks instead of a live user.

In [ ]:
# Setup (run once)
!git clone https://github.com/kgrid-objects/FAIR-DO-Workshop.git

%cd /content/FAIR-DO-Workshop/collection/HBOT-Regimen-Burden-KO
!npm install

%cd /content/FAIR-DO-Workshop/HBOT3-KA
!node -v

In [ ]:
%%bash
node - <<'NODE'
const { executeKnowledgeAssembly } = require('./src/orchestrator');

const request = {
  request_id: 'req-colab-demo',
  requested_at: '2026-09-24T12:00:00Z',
  index_time: '2026-09-24T10:00:00Z',
  subject_binding: {
    subject_identifier: { system: 'https://example.org/mrn', value: 'MRN-001' },
    ulcer_identifier: { system: 'https://example.org/ulcer', value: 'ULCER-001' },
    care_episode_identifier: { system: 'https://example.org/episode', value: 'EPISODE-001' },
    source_evidence: {}
  },
  hbot_case_assertions: {
    dfu_confirmed: { value: true, source_evidence: {} },
    acute_surgical_intervention: { value: true, source_evidence: {} },
    not_healed_after_30_days: { value: false, source_evidence: {} }
  },
  margolis_first_visit_assessment: {
    wound_area: { value: 1, ucum_code: 'cm2' },
    wound_duration: { value: 4, ucum_code: 'wk' },
    first_visit_at: '2026-09-10T09:00:00Z',
    first_visit_attested: true
  }
};

// Scripted answers standing in for a live user answering the Wagner and
// Burden KOs' own interactive Questionnaire Logic capability.
const wagnerAnswers = { Q01: false, Q02: false, Q06: true, Q07: true, Q03: true };
const burdenAnswers = {
  Q01: 'https://kgrid.org/cks/dfu-hbot-burden/providers/e-001',
  Q02: 5,
  Q03: 10,
  Q04: 'none'
};

executeKnowledgeAssembly(request, {
  wagnerAskYesNo: async (question) => wagnerAnswers[question.id],
  burdenAskQuestion: async (question) => burdenAnswers[question.id]
}).then((result) => {
  console.log(JSON.stringify(result, null, 2));
});
NODE